# 🛫 Model 3: Isolation Forest Anomaly Detection
### KQ Water Bottling Plant (Project Kifaru) — Machine Learning Intelligence
---
**Objective:** Flag unusual transaction patterns.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from sklearn.ensemble import IsolationForest

# Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
print("Libraries successfully imported!")

## 1. Load Dataset from GitHub
We extract the historical transactional logs directly from the system's database CSV export.

In [ ]:
dataset_url = "https://raw.githubusercontent.com/aawbed/kq-sales-order-intelligence/main/kq_historical_sales_2025_2026.csv"
df_orders = pd.read_csv(dataset_url)

df_orders['date'] = pd.to_datetime(df_orders['Date'])
df_orders = df_orders.rename(columns={
    'Order ID': 'order_id',
    'Customer Name': 'customer_name',
    'Account Type': 'account_type',
    'Product Name': 'product_name',
    'Quantity': 'quantity',
    'Unit Price (KSh)': 'unit_price_kes',
    'Total Amount (KSh)': 'total_amount_kes',
    'Total Litres': 'total_litres'
})

print(f"Dataset successfully loaded!\nTotal Records: {len(df_orders)}")
df_orders.head()

## 2. Model Training & Evaluation

In [ ]:
# Feature Engineering
cust_freq = df_orders.groupby('customer_name')['order_id'].transform('count')
df_orders['customer_order_freq'] = cust_freq
features = df_orders[['quantity', 'total_amount_kes', 'customer_order_freq']]

# Train Isolation Forest
iso_forest = IsolationForest(contamination=0.04, random_state=42)
df_orders['anomaly_pred'] = iso_forest.fit_predict(features)
df_orders['anomaly_score'] = iso_forest.decision_function(features)

df_orders['is_flagged'] = df_orders['anomaly_pred'] == -1
flagged_orders = df_orders[df_orders['is_flagged']]

print(f"Total Orders Evaluated: {len(df_orders)}")
print(f"Flagged Anomalous Orders: {len(flagged_orders)}\n")

# Plot
plt.figure(figsize=(14, 6))
plt.scatter(
    df_orders[~df_orders['is_flagged']]['quantity'],
    df_orders[~df_orders['is_flagged']]['total_amount_kes'],
    c='#1c7ed6', alpha=0.5, label='Normal Orders', s=45
)
plt.scatter(
    flagged_orders['quantity'],
    flagged_orders['total_amount_kes'],
    c='#c8102e', alpha=0.9, label='Flagged Anomalies', s=120, edgecolors='black', marker='X'
)

plt.title("Isolation Forest Anomaly Detection", fontweight='bold')
plt.xlabel("Order Quantity (Cases/Units)")
plt.ylabel("Total Order Value (KSh)")
plt.legend()
plt.show()

print("\n=== Top Flagged Anomalies ===")
display(flagged_orders[['order_id', 'date', 'customer_name', 'product_name', 'quantity', 'total_amount_kes', 'anomaly_score']].head())